In [1]:
# SYSTEM
import os

# DOCX Processing
from docx import Document

# LangChain & FAISS
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import FAISS
from langchain.embeddings import HuggingFaceEmbeddings

# Sentence Transformers for embedding models
from sentence_transformers import SentenceTransformer

# Hugging Face login for gated models
from huggingface_hub import login

# Transformers & LangChain LLM
from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline
from langchain.llms import HuggingFacePipeline

# LangChain retrieval Q&A
from langchain.chains import RetrievalQA


In [2]:
from huggingface_hub import login
login()


In [3]:
def load_all_docx(folder_path):
    all_texts = []
    for filename in os.listdir(folder_path):
        if filename.endswith(".docx"):
            file_path = os.path.join(folder_path, filename)
            doc = Document(file_path)
            text = "\n".join([para.text for para in doc.paragraphs if para.text.strip()])
            all_texts.append(text)
    return all_texts

In [4]:
# Example Windows path
folder_path = r"C:\USDA\dataset"
texts = load_all_docx(folder_path)


In [5]:
splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)
documents = splitter.create_documents(texts)

print(f"✅ Loaded and split {len(documents)} document chunks.")


✅ Loaded and split 73 document chunks.


In [6]:
embedding_model = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")
db = FAISS.from_documents(documents, embedding_model)
db.save_local("faiss_hf_index")

print("✅ FAISS index created and saved locally.")


C:\Users\axi034\AppData\Local\Temp\ipykernel_25564\7504430.py:1: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the :class:`~langchain-huggingface package and should be used instead. To use it run `pip install -U :class:`~langchain-huggingface` and import as `from :class:`~langchain_huggingface import HuggingFaceEmbeddings``.
  embedding_model = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")


✅ FAISS index created and saved locally.


In [7]:
model_name = "mistralai/Mistral-7B-Instruct-v0.1"  # Change if using another

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(model_name, device_map="auto")

pipe = pipeline(
    "text-generation",
    model=model,
    tokenizer=tokenizer,
    device_map="auto"
)

llm = HuggingFacePipeline(pipeline=pipe)

print("✅ Model loaded.")


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

Some parameters are on the meta device because they were offloaded to the cpu and disk.
Device set to use cuda:0


✅ Model loaded.


C:\Users\axi034\AppData\Local\Temp\ipykernel_25564\2887606557.py:13: LangChainDeprecationWarning: The class `HuggingFacePipeline` was deprecated in LangChain 0.0.37 and will be removed in 1.0. An updated version of the class exists in the :class:`~langchain-huggingface package and should be used instead. To use it run `pip install -U :class:`~langchain-huggingface` and import as `from :class:`~langchain_huggingface import HuggingFacePipeline``.
  llm = HuggingFacePipeline(pipeline=pipe)


In [8]:
from langchain.memory import ConversationBufferMemory
from langchain.chains import ConversationalRetrievalChain

memory = ConversationBufferMemory(memory_key="chat_history", return_messages=True)


C:\Users\axi034\AppData\Local\Temp\ipykernel_25564\3011284198.py:4: LangChainDeprecationWarning: Please see the migration guide at: https://python.langchain.com/docs/versions/migrating_memory/
  memory = ConversationBufferMemory(memory_key="chat_history", return_messages=True)


In [10]:

retriever = db.as_retriever(search_type="similarity", search_kwargs={"k": 4})

rag_chain = ConversationalRetrievalChain.from_llm(
    llm=llm,
    retriever=retriever,
    memory=memory,
    return_source_documents=True
)


In [11]:
chat_history = []

while True:
    query = input("\n📝 Enter your question (or type 'exit' to quit): ")
    if query.lower() == 'exit':
        print("Session ended.")
        break

    result = rag_chain({"question": query, "chat_history": chat_history})
    
    print("\n🎯 Answer:\n", result["answer"])
    
    # Append current Q&A to chat_history
    chat_history.append((query, result["answer"]))
    
    again = input("\n❓ Do you want to ask another question? (yes/no): ")
    if again.lower() not in ['yes', 'y']:
        print(" Session ended.")
        break



📝 Enter your question (or type 'exit' to quit):  Why is the Asian Citrus Psyllid considered an efficient vector for HLB?


C:\Users\axi034\AppData\Local\Temp\ipykernel_25564\1240231493.py:9: LangChainDeprecationWarning: The method `Chain.__call__` was deprecated in langchain 0.1.0 and will be removed in 1.0. Use :meth:`~invoke` instead.
  result = rag_chain({"question": query, "chat_history": chat_history})
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.


ValueError: Got multiple output keys: dict_keys(['answer', 'source_documents']), cannot determine which to store in memory. Please set the 'output_key' explicitly.